In [1]:
!pip install torchinfo

In [2]:
from torchvision import models
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import requests
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
projection_size = 32
backbone = models.resnet18(num_classes=projection_size)
summary(backbone)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            9,408
├─BatchNorm2d: 1-2                       128
├─ReLU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  36,864
│    │    └─BatchNorm2d: 3-2             128
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  36,864
│    │    └─BatchNorm2d: 3-5             128
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  36,864
│    │    └─BatchNorm2d: 3-7             128
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  36,864
│    │    └─BatchNorm2d: 3-10            128
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11                 73,728

In [4]:
class ClassifierHead(nn.Module):
    """single-layer linear task head for few-shot classification"""
    def __init__(self,
                 num_classes: int, # number of output neurons for classification
                 input_size: int = projection_size, # size of the feature vector from the backbone
                 dropout: bool = True, # important to avoid overfitting on tiny classification tasks
                ):
        super().__init__()

        self.fc = nn.Linear(input_size, num_classes)
        self.relu = nn.ReLU()

        if dropout:
            self.dropout = nn.Dropout(0.5)
        else:
            self.dropout = nn.Identity()

        # self.to(device)

    def forward(self, x):
        # assume x is already unactivated feature logits, so we activate it first
        x = self.fc(self.relu(self.dropout(x)))

        return x

In [5]:
# label_to_idx = {label: idx for idx, label in enumerate(self.data['class'].unique())}
num_classes = 11

head = ClassifierHead(num_classes=num_classes)
summary(head)

Layer (type:depth-idx)                   Param #
ClassifierHead                           --
├─Linear: 1-1                            363
├─ReLU: 1-2                              --
├─Dropout: 1-3                           --
Total params: 363
Trainable params: 363
Non-trainable params: 0

In [6]:
model = nn.Sequential(
    backbone,
    head
)
summary(model)

Layer (type:depth-idx)                        Param #
Sequential                                    --
├─ResNet: 1-1                                 --
│    └─Conv2d: 2-1                            9,408
│    └─BatchNorm2d: 2-2                       128
│    └─ReLU: 2-3                              --
│    └─MaxPool2d: 2-4                         --
│    └─Sequential: 2-5                        --
│    │    └─BasicBlock: 3-1                   73,984
│    │    └─BasicBlock: 3-2                   73,984
│    └─Sequential: 2-6                        --
│    │    └─BasicBlock: 3-3                   230,144
│    │    └─BasicBlock: 3-4                   295,424
│    └─Sequential: 2-7                        --
│    │    └─BasicBlock: 3-5                   919,040
│    │    └─BasicBlock: 3-6                   1,180,672
│    └─Sequential: 2-8                        --
│    │    └─BasicBlock: 3-7                   3,673,088
│    │    └─BasicBlock: 3-8                   4,720,640
│    └─AdaptiveA

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
BATCH_SIZE = 64

def train_supervised(
        model: nn.Module,
        train_loader: torch.utils.data.DataLoader,
        num_epochs = 10,
        lr = 0.1,
        val_loader = None,
        l2_reg = 0.0,
        loss_fn = nn.CrossEntropyLoss()
    ):

    opt = torch.optim.SGD(model.parameters(), momentum=0.9, lr=lr, weight_decay=l2_reg)

    torch.manual_seed(42)

    for e in range(num_epochs):

        # ========= TRAINING =========
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        for batch_id, (x, y, _) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            opt.zero_grad()
            pred = model(x)

            batch_loss = loss_fn(pred, y)
            batch_loss.backward()
            opt.step()

            total_loss += batch_loss.item() * x.size(0)
            predicted_labels = pred.argmax(dim=1)
            total_correct += (predicted_labels == y).sum().item()
            total_samples += y.size(0)

            print(f"Epoch {e+1}/{num_epochs} | "
                    f"Batch {batch_id+1}/{len(train_loader)} | "
                    f"Loss: {batch_loss.item():.4f} | "
                    f"Acc: {total_correct/total_samples:.4f}")

        train_loss = total_loss / total_samples
        train_acc = total_correct / total_samples

        # ========= VALIDATION =========
        if val_loader:
            model.eval()
            val_loss = 0
            val_correct = 0
            val_samples = 0

            with torch.no_grad():
                for x, y, _ in val_loader:
                    x, y = x.to(device), y.to(device)

                    pred = model(x)
                    loss = loss_fn(pred, y)

                    val_loss += loss.item() * x.size(0)
                    predicted_labels = pred.argmax(dim=1)
                    val_correct += (predicted_labels == y).sum().item()
                    val_samples += y.size(0)

            val_loss /= val_samples
            val_acc = val_correct / val_samples

            print(f"Epoch {e+1}/{num_epochs} | "
                f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        else:
            print(f"Epoch {e+1}/{num_epochs} | "
                f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")


    return

cuda


In [8]:
class SportsDataset(Dataset):
    def __init__(self, csv_file, labelled=True, transform=None):
        """
        Args:
            csv_file (str): Path to CSV file with annotations.
            transform (callable, optional): Transform to be applied on an image.
        """
        self.data = pd.read_csv(csv_file)
        self.labelled = labelled                # TO ALLOW FOR BOTH SUPERCLASS AND CLASS OR ONLY SUPERCLASS
        self.transform = transform

        # # Build mappings for labels/superclasses → integers
        # self.label_to_idx = {label: idx for idx, label in enumerate(self.data['class'].unique())}
        # self.super_to_idx = {sup: idx for idx, sup in enumerate(self.data['superclass'].unique())}
        # print(self.label_to_idx)
        # print(self.super_to_idx)

        self.data["super_idx"] = self.data["superclass"].apply(lambda x: superclassDict[x])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        path = row['filepath']
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.labelled:
            # label = self.label_to_idx[row['class']]
            # superclass = self.super_to_idx[row['superclass']]
            label = row['class']
            superclass = row['super_idx']

            return image, superclass, label
        else:
            superclass = row['superclass']
            return image, superclass

In [9]:
superclassDict = {
    'precision & target sports': 0,
    'water': 1,
    'field & team ball sports': 2,
    'court ball sports': 3,
    'combat & strength sports': 4,
    'equestrian & animal sports': 5,
    'ice & snow sports': 6,
    'motor & wheel racing': 7,
    'gymnastics': 8,
    'aerial': 9,
    'track & field athletics': 10
}

In [10]:
model = model.to(device)


In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

superclass_dataset = SportsDataset(
    csv_file='/content/drive/MyDrive/copy.csv', labelled=True, transform=transform)

superclass_loader = DataLoader(
    superclass_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
train_supervised(model, superclass_loader)

Epoch 1/10 | Batch 1/375 | Loss: 2.3626 | Acc: 0.1719
Epoch 1/10 | Batch 2/375 | Loss: 2.3468 | Acc: 0.1328
Epoch 1/10 | Batch 3/375 | Loss: 2.2971 | Acc: 0.1667
Epoch 1/10 | Batch 4/375 | Loss: 2.4203 | Acc: 0.1523
Epoch 1/10 | Batch 5/375 | Loss: 2.3773 | Acc: 0.1437
Epoch 1/10 | Batch 6/375 | Loss: 2.3352 | Acc: 0.1536
Epoch 1/10 | Batch 7/375 | Loss: 2.3225 | Acc: 0.1540
Epoch 1/10 | Batch 8/375 | Loss: 2.3828 | Acc: 0.1504
Epoch 1/10 | Batch 9/375 | Loss: 2.3336 | Acc: 0.1493
Epoch 1/10 | Batch 10/375 | Loss: 2.4105 | Acc: 0.1469
Epoch 1/10 | Batch 11/375 | Loss: 2.4000 | Acc: 0.1449
Epoch 1/10 | Batch 12/375 | Loss: 2.3434 | Acc: 0.1432
Epoch 1/10 | Batch 13/375 | Loss: 2.2961 | Acc: 0.1442
Epoch 1/10 | Batch 14/375 | Loss: 2.2949 | Acc: 0.1507
Epoch 1/10 | Batch 15/375 | Loss: 2.3321 | Acc: 0.1531
Epoch 1/10 | Batch 16/375 | Loss: 2.3339 | Acc: 0.1543
Epoch 1/10 | Batch 17/375 | Loss: 2.3334 | Acc: 0.1507
Epoch 1/10 | Batch 18/375 | Loss: 2.3382 | Acc: 0.1510
Epoch 1/10 | Batch 